**Step # 1 : Data Ingestion & Preprocessing**

In [13]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters

In [3]:
import os

os.makedirs('rag_project/blogs', exist_ok=True)

print('Directory structure created:')
print('rag_project/')
print('└── blogs/  <- place your .txt files here')

Directory structure created:
rag_project/
└── blogs/  <- place your .txt files here


In [7]:
from google.colab import files
import shutil

print('Upload your .txt blog files...')
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f'rag_project/blogs/{filename}')
    print(f'Moved {filename} to rag_project/blogs/{filename}')

Upload your .txt blog files...


Saving Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt to Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
Moved Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt to rag_project/blogs/Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt


In [8]:
import os
import re

BLOGS_DIR = 'rag_project/blogs'

def clean_text(text):
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()
    return text

def load_blogs(directory):
    blogs = []
    files = sorted([f for f in os.listdir(directory) if f.endswith('.txt')])

    if not files:
        print('No .txt files found in', directory)
        return blogs

    for filename in files:
        filepath = os.path.join(directory, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        cleaned = clean_text(raw_text)
        lines = cleaned.splitlines()
        title = filename
        if lines and lines[0].lower().startswith('title:'):
            title = lines[0].replace('Title:', '').replace('title:', '').strip()

        blogs.append({'filename': filename, 'title': title, 'content': cleaned})
        print(f'Loaded: {filename} | Title: "{title}" | Length: {len(cleaned)} chars')

    return blogs

blogs = load_blogs(BLOGS_DIR)
print(f'\nTotal blogs loaded: {len(blogs)}')

Loaded: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt | Title: "Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt" | Length: 6761 chars

Total blogs loaded: 1


In [9]:
for blog in blogs:
    print(f"\n{'='*60}")
    print(f"File    : {blog['filename']}")
    print(f"Title   : {blog['title']}")
    print(f"Length  : {len(blog['content'])} characters | {len(blog['content'].split())} words")
    print(f"\nPreview (first 500 chars):")
    print(blog['content'][:500])
    print('...')


File    : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
Title   : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
Length  : 6761 characters | 1034 words

Preview (first 500 chars):
Top 10 life-changing personal development goals you need10 life-changing personal development goals you need
February 4, 2025
Share
Success and achievement do not happen by chance. They are built on the understanding of growth and the pursuit of meaningful goals. 
The purpose of personal development is the stepping stone to improving your mood, personality and overall quality of life. Whether you are a self-improvement enthusiast, a student looking to achieve your goals or a professional looking
...


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

def chunk_blogs(blogs):
    all_chunks = []
    for blog in blogs:
        chunks = text_splitter.split_text(blog['content'])
        for i, chunk_text in enumerate(chunks):
            doc = Document(
                page_content=chunk_text,
                metadata={
                    'source': blog['filename'],
                    'title': blog['title'],
                    'chunk_id': i
                }
            )
            all_chunks.append(doc)
        print(f"'{blog['filename']}' -> {len(chunks)} chunks")
    return all_chunks

all_chunks = chunk_blogs(blogs)
print(f'\nTotal chunks created: {len(all_chunks)}')

'Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt' -> 10 chunks

Total chunks created: 10


In [15]:
print(f"{'#':<5} {'Source':<15} {'Chunk ID':<10} {'Words':<10} {'Preview'}")
print('-' * 75)
for i, doc in enumerate(all_chunks):
    word_count = len(doc.page_content.split())
    preview = doc.page_content[:55].replace('\n', ' ')
    print(f"{i:<5} {doc.metadata['source']:<15} {doc.metadata['chunk_id']:<10} {word_count:<10} {preview}...")

#     Source          Chunk ID   Words      Preview
---------------------------------------------------------------------------
0     Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt 0          152        Top 10 life-changing personal development goals you nee...
1     Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt 1          62         Ways to achieve this: * Keep a journal to track your th...
2     Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt 2          77         Life is full of surprises and changes, but adversity ma...
3     Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt 3          143        Effective time management is essential for creating spa...
4     Top 10 life-changing personal development goals you need10 life-ch

In [16]:
INSPECT_INDEX = 0
doc = all_chunks[INSPECT_INDEX]
print(f"Chunk #{INSPECT_INDEX}")
print(f"Source   : {doc.metadata['source']}")
print(f"Title    : {doc.metadata['title']}")
print(f"Chunk ID : {doc.metadata['chunk_id']}")
print(f"Words    : {len(doc.page_content.split())}")
print(f"\nFull Content:")
print(doc.page_content)

Chunk #0
Source   : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
Title    : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
Chunk ID : 0
Words    : 152

Full Content:
Top 10 life-changing personal development goals you need10 life-changing personal development goals you need
February 4, 2025
Share
Success and achievement do not happen by chance. They are built on the understanding of growth and the pursuit of meaningful goals. 
The purpose of personal development is the stepping stone to improving your mood, personality and overall quality of life. Whether you are a self-improvement enthusiast, a student looking to achieve your goals or a professional looking to reach the next level, these goals can inspire change.
Where do you want to start? Here are ten personal development goals that can change your life.
1. Cultivate self-awareness 
Self-awareness is 

In [17]:
word_counts = [len(doc.page_content.split()) for doc in all_chunks]
print('Chunk Word Count Statistics')
print(f'  Min   : {min(word_counts)} words')
print(f'  Max   : {max(word_counts)} words')
print(f'  Avg   : {sum(word_counts) / len(word_counts):.1f} words')
print(f'  Total : {len(word_counts)} chunks')
print('\nStep 1 Complete - chunks are ready for embedding in Step 2!')

Chunk Word Count Statistics
  Min   : 28 words
  Max   : 159 words
  Avg   : 110.5 words
  Total : 10 chunks

Step 1 Complete - chunks are ready for embedding in Step 2!


# Step 2: Embedding + Vector Store

Goals:
- Use `sentence-transformers` (all-MiniLM-L6-v2) to generate embeddings
- Store embeddings in FAISS with document metadata
- Save FAISS index to disk for persistence
- Reload and verify the saved index

 ## 2.1 : Load the Embedding Model

In [18]:
!pip install -q langchain-text-splitters \
                sentence-transformers faiss-cpu

In [19]:
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('All imports successful')

All imports successful


In [20]:
BLOGS_DIR = 'rag_project/blogs'

def clean_text(text):
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def load_blogs(directory):
    blogs = []
    files = sorted([f for f in os.listdir(directory) if f.endswith('.txt')])
    for filename in files:
        with open(os.path.join(directory, filename), 'r', encoding='utf-8') as f:
            raw = f.read()
        cleaned = clean_text(raw)
        lines = cleaned.splitlines()
        title = lines[0].replace('Title:', '').strip() if lines and lines[0].lower().startswith('title:') else filename
        blogs.append({'filename': filename, 'title': title, 'content': cleaned})
        print(f'Loaded: {filename} | {len(cleaned.split())} words')
    return blogs

def chunk_blogs(blogs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=150,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    chunks = []
    for blog in blogs:
        for i, text in enumerate(splitter.split_text(blog['content'])):
            chunks.append(Document(
                page_content=text,
                metadata={'source': blog['filename'], 'title': blog['title'], 'chunk_id': i}
            ))
        print(f"Chunked: {blog['filename']} -> {i+1} chunks")
    return chunks

blogs      = load_blogs(BLOGS_DIR)
all_chunks = chunk_blogs(blogs)
print(f'\nTotal chunks ready for embedding: {len(all_chunks)}')

Loaded: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt | 1034 words
Chunked: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt -> 10 chunks

Total chunks ready for embedding: 10


In [21]:
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

print(f'Loading embedding model: {EMBEDDING_MODEL}')
print('This may take a minute on first run (downloads ~80 MB)...')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},   # change to 'cuda' if GPU is available
    encode_kwargs={'normalize_embeddings': True}  # normalise for cosine similarity
)

print('Embedding model loaded successfully')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
This may take a minute on first run (downloads ~80 MB)...


/tmp/ipykernel_935/117137135.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


In [22]:
# Sanity check: embed one sentence and inspect the vector
test_vector = embeddings.embed_query('What is personal development?')

print(f'Vector dimensions : {len(test_vector)}')
print(f'First 5 values    : {[round(v, 4) for v in test_vector[:5]]}')
print(f'Vector type       : {type(test_vector)}')
print('Embedding model is working correctly')

Vector dimensions : 384
First 5 values    : [0.0337, 0.0303, -0.0209, 0.0224, 0.0056]
Vector type       : <class 'list'>
Embedding model is working correctly


## 2.2 : Build FAISS Vector Store

In [24]:
print(f'Building FAISS index from {len(all_chunks)} chunks...')
print('Embedding all chunks — this may requires time upto 1 Minutes. Be Patient')

vector_store = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

print(f'\nFAISS index built successfully')
print(f'Total vectors stored : {vector_store.index.ntotal}')
print(f'Vector dimensions    : {vector_store.index.d}')

Building FAISS index from 10 chunks...
Embedding all chunks — this may requires time upto 1 Minutes. Be Patient

FAISS index built successfully
Total vectors stored : 10
Vector dimensions    : 384


In [25]:
# Quick retrieval test before saving
test_query = 'How do I improve my communication skills?'
results = vector_store.similarity_search(test_query, k=2)

print(f'Test query: "{test_query}"')
print(f'Top {len(results)} results:\n')
for i, doc in enumerate(results):
    print(f'  Result {i+1}')
    print(f'  Source   : {doc.metadata["source"]}')
    print(f'  Title    : {doc.metadata["title"]}')
    print(f'  Chunk ID : {doc.metadata["chunk_id"]}')
    print(f'  Preview  : {doc.page_content[:150].replace(chr(10), " ")}...')
    print()

Test query: "How do I improve my communication skills?"
Top 2 results:

  Result 1
  Source   : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
  Title    : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
  Chunk ID : 3
  Preview  : Effective time management is essential for creating space for growth and balancing priorities. Ways to achieve this: * Use a planner, time-blocking ap...

  Result 2
  Source   : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
  Title    : Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt
  Chunk ID : 4
  Preview  : * Ask thoughtful questions to demonstrate genuine interest in the conversation. Why it matters: Effective communication builds trust and connections, ...



## 2.3 Save FAISS Index to Disk

In [26]:
FAISS_INDEX_PATH = 'rag_project/faiss_index'

os.makedirs(FAISS_INDEX_PATH, exist_ok=True)
vector_store.save_local(FAISS_INDEX_PATH)

# Confirm files were written
saved_files = os.listdir(FAISS_INDEX_PATH)
print(f'FAISS index saved to: {FAISS_INDEX_PATH}/')
print(f'Files created: {saved_files}')
# FAISS saves two files:
#   index.faiss  -> the actual vector index (binary)
#   index.pkl    -> document metadata + text (pickle)

FAISS index saved to: rag_project/faiss_index/
Files created: ['index.pkl', 'index.faiss']


In [27]:
print('Reloading FAISS index from disk...')

reloaded_store = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True  # required for pickle-based reload
)

print(f'Index reloaded successfully')
print(f'Vectors in reloaded index: {reloaded_store.index.ntotal}')

Reloading FAISS index from disk...
Index reloaded successfully
Vectors in reloaded index: 10


# Step 3: LangChain Integration
Goals:
- Create a FAISS Retriever from the saved index
- Build a prompt template that injects retrieved context
- Load a HuggingFace LLM (flan-t5-base — instruction-tuned, better than GPT-2)
- Wire everything into a RetrievalQA chain
- Test with sample queries

In [37]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters \
                sentence-transformers faiss-cpu transformers accelerate

In [43]:
import os
import re
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate           # moved to langchain_core in 1.x
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print('All imports successful')

All imports successful


## 3.1 Reload FAISS Index + Embedding Model

In [44]:
FAISS_INDEX_PATH = 'rag_project/faiss_index'
EMBEDDING_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'

print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print('Reloading FAISS index...')
vector_store = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f'FAISS index loaded | Vectors: {vector_store.index.ntotal}')

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reloading FAISS index...
FAISS index loaded | Vectors: 10


In [45]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# Quick test — check what gets retrieved
test_docs = retriever.invoke('How do I manage my time better?')
print(f'Retrieved {len(test_docs)} chunks for test query\n')
for i, doc in enumerate(test_docs):
    print(f'Chunk {i+1} | Source: {doc.metadata["source"]} | Chunk ID: {doc.metadata["chunk_id"]}')
    print(f'  {doc.page_content[:120].replace(chr(10), " ")}...')
    print()

Retrieved 3 chunks for test query

Chunk 1 | Source: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt | Chunk ID: 3
  Effective time management is essential for creating space for growth and balancing priorities. Ways to achieve this: * U...

Chunk 2 | Source: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt | Chunk ID: 2
  Life is full of surprises and changes, but adversity makes you rise above and triumph over adversity. Ways to achieve th...

Chunk 3 | Source: Top 10 life-changing personal development goals you need10 life-changing personal development goals you need.txt | Chunk ID: 7
  * Stay curious and engage in activities that challenge your current skills. Why it matters: Continuous learning will bro...



In [46]:
PROMPT_TEMPLATE = """You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say: I could not find this in the provided content.
Keep your answer concise and factual.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=['context', 'question']
)
print('Prompt template ready | Variables:', prompt.input_variables)

Prompt template ready | Variables: ['context', 'question']


In [50]:
from transformers import AutoModelForCausalLM   # add this if not in imports

LLM_MODEL = 'gpt2'
print(f'Loading {LLM_MODEL}...')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model     = AutoModelForCausalLM.from_pretrained(LLM_MODEL)   # ✅ CausalLM not Seq2SeqLM

hf_pipe   = pipeline(
    task='text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.3,
    do_sample=True,
    pad_token_id=50256
)
llm = HuggingFacePipeline(pipeline=hf_pipe)
print('LLM ready: gpt2')

Loading gpt2...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready: gpt2


/tmp/ipykernel_935/1197754685.py:18: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipe)


In [51]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {
        'context' : retriever | RunnableLambda(format_docs),
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('LCEL RAG chain built')
print('Flow: query -> retriever -> prompt -> flan-t5 -> answer')

LCEL RAG chain built
Flow: query -> retriever -> prompt -> flan-t5 -> answer


In [52]:
def ask(query):
    print(f'Question : {query}')
    print('-' * 60)
    answer      = rag_chain.invoke(query)
    source_docs = retriever.invoke(query)
    sources     = list({doc.metadata['source'] for doc in source_docs})
    print(f'Answer   : {answer}')
    print(f'Sources  : {sources}')
    print()
    return answer

In [53]:
# Sample Query 1
ask('What are the ways to improve communication skills?')

Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : What are the ways to improve communication skills?
------------------------------------------------------------
Answer   : You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say: I could not find this in the provided content.
Keep your answer concise and factual.

Context:
Effective time management is essential for creating space for growth and balancing priorities.
Ways to achieve this:
* Use a planner, time-blocking app, or calendar to organize your day.
* Create a priority list and complete the highest-value tasks first.
* Eliminate procrastination by breaking down essential tasks into smaller, more manageable pieces. Manageable.
Why it matters:
Managing your time effectively will increase productivity, reduce stress, and help you focus on what matters most.
4. Improve your communication skills
Communication is the foundation of successful relationships, whether personal or professional. From listening skills to express

'You are a helpful assistant. Use ONLY the context below to answer.\nIf the answer is not in the context, say: I could not find this in the provided content.\nKeep your answer concise and factual.\n\nContext:\nEffective time management is essential for creating space for growth and balancing priorities.\nWays to achieve this:\n* Use a planner, time-blocking app, or calendar to organize your day.\n* Create a priority list and complete the highest-value tasks first.\n* Eliminate procrastination by breaking down essential tasks into smaller, more manageable pieces. Manageable.\nWhy it matters:\nManaging your time effectively will increase productivity, reduce stress, and help you focus on what matters most.\n4. Improve your communication skills\nCommunication is the foundation of successful relationships, whether personal or professional. From listening skills to expressing ideas clearly, improving these skills will change the way you communicate with others.\nWays to achieve this: \n* Li

In [54]:
ask('What is retrieval augmented generation?')

Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : What is retrieval augmented generation?
------------------------------------------------------------
Answer   : You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say: I could not find this in the provided content.
Keep your answer concise and factual.

Context:
Ways to achieve this:
* Keep a journal to track your thoughts and actions every day.
* Practice mindfulness through meditation or mindfulness exercises.
* Regularly examine your emotions and see how they affect your actions, decisions, and behaviours.
Why its important:
Self-awareness can help you increase your emotional quotient (EQ) and develop better relationships and a happier life.
2. Find happiness

Life is full of surprises and changes, but adversity makes you rise above and triumph over adversity.
Ways to achieve this:
* Practice gratitude by listing three things you are grateful for each day.
* Learn stress management techniques such as deep breathing or m

'You are a helpful assistant. Use ONLY the context below to answer.\nIf the answer is not in the context, say: I could not find this in the provided content.\nKeep your answer concise and factual.\n\nContext:\nWays to achieve this:\n* Keep a journal to track your thoughts and actions every day.\n* Practice mindfulness through meditation or mindfulness exercises.\n* Regularly examine your emotions and see how they affect your actions, decisions, and behaviours.\nWhy its important:\nSelf-awareness can help you increase your emotional quotient (EQ) and develop better relationships and a happier life.\n2. Find happiness\n\nLife is full of surprises and changes, but adversity makes you rise above and triumph over adversity.\nWays to achieve this:\n* Practice gratitude by listing three things you are grateful for each day.\n* Learn stress management techniques such as deep breathing or meditation.\n* Surround yourself with supportive friends or mentors.\nWhy it matters:\nResilience improves 

In [55]:
# Run this in Colab to zip and download the index
import shutil
shutil.make_archive('faiss_index', 'zip', 'rag_project/faiss_index')

from google.colab import files
files.download('faiss_index.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>